# Exploring the workshop dataset

A short tour of the `wxpost` package and the WACCM-X file we use for the workshop. Run the cells top-to-bottom and look at what the model says. Two of the things you'll see in here are wrong on purpose — they are the workshop exercises. Don't fix them in this notebook; let Claude do that in your session.

If you see `FileNotFoundError`, you're not running on Derecho (or `/glade/campaign` isn't mounted).

## 0. Make sure `wxpost` is importable

If you opened this notebook in a kernel that doesn't have the package installed yet, the cell below pip-installs it from the repo. Safe to re-run.

In [ ]:
import sys, subprocess
try:
    import wxpost  # noqa: F401
    print('wxpost already importable from', wxpost.__file__)
except ModuleNotFoundError:
    print('installing wxpost from ../ into', sys.executable)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '..'])
    import wxpost  # noqa: F401
    print('installed at', wxpost.__file__)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from wxpost import open_waccmx, to_pressure, to_height, zonal_mean
from wxpost.coords import pressure_pa, height_m

## 1. Open the file

One month of WACCM-X FXSD output, monthly mean (`cam.h0`), Jan 2020, on the f19 grid (96 × 144) with 145 levels. The full file has ~360 variables; we only pull the ones we'll use.

In [ ]:
WACCMX = (
    "/glade/campaign/hao/itmodel/joemci/archive/"
    "f.e22.FXSD.f19_f19_mg17.001/atm/hist/2020/"
    "f.e22.FXSD.f19_f19_mg17.001.cam.h0.2020-01.nc"
)

ds = open_waccmx(WACCMX, variables=("T", "U", "V", "TElec", "EDens", "Z3"))
ds

## 2. The hybrid sigma-pressure vertical coordinate

Model levels (`lev`) aren't pressures themselves — they're indices into a hybrid sigma-pressure formula:

$$
p(\mathrm{lev}, \mathrm{lat}, \mathrm{lon}) = \mathrm{hyam}(\mathrm{lev}) \cdot P_0 + \mathrm{hybm}(\mathrm{lev}) \cdot P_S(\mathrm{lat}, \mathrm{lon})
$$

`wxpost.coords.pressure_pa` does that. The pressure spans ~13 orders of magnitude from the top of the model to the surface.

In [ ]:
p = pressure_pa(ds)
z = height_m(ds)

p_profile = p.mean(dim=["time", "lat", "lon"]).values
z_profile = z.mean(dim=["time", "lat", "lon"]).values

print(f"top of model:   {p_profile.min():.2e} Pa   ({z_profile.max()/1000:.0f} km)")
print(f"surface:        {p_profile.max():.2e} Pa   ({z_profile.min()/1000:.2f} km)")
print(f"number of levels: {len(p_profile)}")

## 3. Surface temperature, January

Plot January monthly-mean surface temperature. NH is in winter, SH is in summer. Look at the asymmetry. Does anything look off?

In [ ]:
T_surf = ds["T"].isel(time=0, lev=-1) - 273.15

fig, ax = plt.subplots(figsize=(11, 4.5))
pcm = ax.pcolormesh(T_surf["lon"], T_surf["lat"], T_surf.values,
                    cmap="RdBu_r", vmin=-50, vmax=40, shading="auto")
ax.set_xlabel("longitude (°E)"); ax.set_ylabel("latitude (°N)")
ax.set_title("Surface T  ·  January 2020")
fig.colorbar(pcm, ax=ax, label="T (°C)")
plt.show()

In [ ]:
# Zonal mean. NH winter should be far colder than SH summer at the same |lat|.
T_zm = T_surf.mean(dim="lon")

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(T_zm.values, T_zm["lat"].values, lw=2)
ax.axhline(0, color="k", lw=0.5, alpha=0.3)
ax.axvline(0, color="k", lw=0.5, alpha=0.3)
ax.set_xlabel("zonal-mean T (°C)"); ax.set_ylabel("latitude (°N)")
ax.set_title("Zonal-mean surface T, January"); ax.grid(alpha=0.3)

print(f"60°N (label):  {float(T_zm.sel(lat=60, method='nearest')):+.1f}°C")
print(f"60°S (label):  {float(T_zm.sel(lat=-60, method='nearest')):+.1f}°C")
plt.show()

## 4. Vertical structure of T and TElec

Neutral temperature `T` and electron temperature `TElec` are equal below ~120 km — electrons are in thermal equilibrium with the neutral gas there. Above ~150 km they diverge sharply: photoionization heats the electrons and they decouple. By 300 km, `TElec ≈ 2 × T`.

In [ ]:
t_prof  = ds["T"].mean(dim=["time", "lat", "lon"]).values
te_prof = ds["TElec"].mean(dim=["time", "lat", "lon"]).values
alt_km  = z_profile / 1000.0

fig, ax = plt.subplots(figsize=(5, 6))
ax.plot(t_prof,  alt_km, label="T (neutral)", lw=2)
ax.plot(te_prof, alt_km, label="TElec", lw=2)
ax.set_xlabel("temperature (K)"); ax.set_ylabel("altitude (km)")
ax.set_title("Global-mean temperature profile")
ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0, 450)
plt.show()

## 5. Interpolate to fixed altitudes

The package's main operations are `to_pressure` (Pa → field) and `to_height` (m → field). Compare `TElec` at 300 km from the two interpolation methods.

From the profile above, the global-mean `TElec` at 300 km is somewhere around 1500–1700 K. Both methods should land near that.

In [ ]:
altitudes = np.array([300_000.0])  # metres

te_default = to_height(ds, "TElec", altitudes)
te_loglin  = to_height(ds, "TElec", altitudes, method="loglinear")

g_default = float(te_default.mean().values)
g_loglin  = float(te_loglin.mean().values)

print(f"to_height(TElec, 300 km) default     :  {g_default:7.1f} K")
print(f"to_height(TElec, 300 km) loglinear   :  {g_loglin:7.1f} K")
print(f"Reading the profile above, you expect about 1500–1700 K.")

## 6. Try it yourself

A handful of things worth poking at:

- Plot `EDens` (electron density) at the F-region peak. WACCM-X-specific.
- Compare `to_pressure(ds, 'T', [1e4, 1e3, 1e2])` between the two methods.
- Take a zonal mean of `TElec` and plot it as `(lat, altitude)`. The auroral signature should jump out.
- Run `pytest` from the shell. Two tests fail. The cells above contain hints about why.